In [2]:
# Notebook setup: import and (optionally) auto-reload local module
import importlib
import time

import locator_time

# If you edit locator_time.py while this notebook is open, re-run this cell to reload.
importlib.reload(locator_time)

print(locator_time.__doc__[:400])

A small, self-contained Python port of ImPlot's time-axis tick locator.

This is based on the logic in `implot.cpp` (function `Locator_Time`) and the
associated helpers in the "Time Ticks and Utils" section.

Goal
----
Given:
  - `t_min` and `t_max` as Unix timestamps in seconds (float or int)
  - `pixels` as the axis pixel length (float or int)

Produce:
  - tick positions (float seconds)
  - lab


In [3]:
# Small helpers to inspect returned ticks
from collections import Counter

def summarize_ticks(ticks):
    c = Counter((t.level, t.major, t.show_label) for t in ticks)
    total = len(ticks)
    level0 = sum(1 for t in ticks if t.level == 0)
    level1 = sum(1 for t in ticks if t.level == 1)
    shown = sum(1 for t in ticks if t.show_label)
    return {
        'total_ticks': total,
        'level0_ticks': level0,
        'level1_ticks': level1,
        'labels_shown': shown,
        'breakdown': dict(c),
    }

def head_ticks(ticks, n=20):
    rows = []
    for t in ticks[:n]:
        tag = f"L{t.level} {'M' if t.major else 'm'}"
        label = t.label if t.show_label else ''
        rows.append((tag, t.pos, label))
    return rows

## Baseline: 1 hour range
This should produce minute/second-ish ticks depending on `pixels` and `max_density`.

In [ ]:
now = time.time()
t_min = now
t_max = now + 3600

ticks = locator_time.locator_time(
    t_min, t_max, pixels=800,
    use_local_time=True,
    max_density=0.5,
    char_px=7.0,
)

summarize_ticks(ticks), head_ticks(ticks, 25)

## Compare pixel widths
Smaller `pixels` should suppress more labels (especially level 0 minor labels).

In [ ]:
for px in (200, 400, 800, 1200):
    ticks_px = locator_time.locator_time(t_min, t_max, pixels=px, use_local_time=True)
    s = summarize_ticks(ticks_px)
    print(f"pixels={px:4d}  total={s['total_ticks']:4d}  shown={s['labels_shown']:4d}  L0={s['level0_ticks']:4d}  L1={s['level1_ticks']:4d}")

## Explore different spans
These cover typical unit transitions (minutes → hours → days → months → years).

In [ ]:
def run_span(span_seconds, pixels=900, title=None):
    t0 = time.time()
    t1 = t0 + span_seconds
    ticks = locator_time.locator_time(t0, t1, pixels=pixels, use_local_time=True)
    s = summarize_ticks(ticks)
    title = title or f"span={span_seconds}s"
    print(f"\n{title} (pixels={pixels})")
    print(f"  total={s['total_ticks']}  shown={s['labels_shown']}")
    print('  first 12:', head_ticks(ticks, 12))

run_span(10, title='10 seconds')
run_span(5 * 60, title='5 minutes')
run_span(6 * 3600, title='6 hours')
run_span(2 * 86400, title='2 days')
run_span(45 * 86400, title='45 days')
run_span(400 * 86400, title='~400 days (year-ish)')
run_span(10 * 365 * 86400, title='~10 years (year locator)')

## ISO-8601 / 24-hour formatting toggles
These flags match the knobs you might want in a UI layer.

In [ ]:
t_min = time.time()
t_max = t_min + 3 * 3600

ticks_default = locator_time.locator_time(t_min, t_max, 800, use_local_time=True, use_24_hour=False, use_iso8601=False)
ticks_iso24 = locator_time.locator_time(t_min, t_max, 800, use_local_time=True, use_24_hour=True, use_iso8601=True)

print('default:', head_ticks(ticks_default, 10))
print('iso+24:', head_ticks(ticks_iso24, 10))

## Notes on fidelity vs ImPlot
- The biggest mismatch is **text width measurement**: ImPlot uses actual font metrics; here we use `len(label) * char_px`.
- If you have access to real text measurement in your UI, replace `estimate_label_width_px(...)` with it for more faithful label suppression.